In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'
!wget $data -O data-week-3.csv

--2026-05-16 08:04:37--  https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 977501 (955K) [text/plain]
Saving to: ‘data-week-3.csv’

data-week-3.csv     100%[===================>] 954.59K  4.37MB/s    in 0.2s    

2026-05-16 08:04:38 (4.37 MB/s) - ‘data-week-3.csv’ saved [977501/977501]



In [3]:
df = pd.read_csv('data-week-3.csv')

# Data preparation

In [4]:
df.columns = df.columns.str.lower()

strings = list(df.dtypes[df.dtypes == 'str'].index)

for col in strings:
    df[col] = df[col].str.lower().str.replace(' ', '_')


In [5]:
#check dtypes to change to apprropriate data types 
df.dtypes

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges            str
churn                   str
dtype: object

In [6]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')

In [7]:
df.totalcharges.dtypes

dtype('float64')

In [8]:
df.totalcharges = df.totalcharges.fillna(0)

In [9]:
df.churn = (df.churn == 'yes').astype(int)

df.churn.head()

0    0
1    0
2    1
3    0
4    1
Name: churn, dtype: int64

# Setting up the validation framework
- train/validation/test

In [10]:
from sklearn.model_selection import  train_test_split

In [11]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
len(df_full_train), len(df_test)

(5634, 1409)

In [12]:
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
len(df_train), len(df_val), len(df_test)

(4225, 1409, 1409)

In [13]:
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)


In [14]:
y_train = df_train.churn.values
y_test = df_test.churn.values
y_val = df_val.churn.values

del df_train['churn']
del df_test['churn']
del df_val['churn']

# EDA

In [15]:
df_full_train = df_full_train.reset_index(drop=True)

### check for missing values

In [16]:
df_full_train.isnull().sum()

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [17]:
#calculating the churn rate

global_churn_rate = round(df_full_train.churn.mean(), 2)
global_churn_rate

np.float64(0.27)

In [18]:
numerical_variables = ['tenure', 'monthlycharges', 'totalcharges']

categorical_variables = [ 'gender', 'seniorcitizen', 'partner', 'dependents',
       'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod']

In [19]:

df.columns = df.columns.str.lower().str.replace(' ', '_')
strings = list(df.dtypes[df.dtypes == 'str'].index)
for col in strings:
    df[col] = df[col].str.lower().str.replace(' ', '_')

In [27]:
df_full_train[categorical_variables]


,gender,seniorcitizen,partner,dependents,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod
0,male,0,yes,yes,yes,no,no,no_internet_service,no_internet_service,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check
1,female,0,no,no,yes,no,dsl,yes,yes,yes,yes,no,yes,one_year,no,credit_card_(automatic)
2,male,0,yes,no,yes,yes,dsl,yes,yes,no,yes,no,no,two_year,no,bank_transfer_(automatic)
3,male,0,yes,yes,yes,yes,dsl,yes,no,yes,yes,yes,yes,one_year,no,electronic_check
4,male,0,no,no,yes,no,dsl,yes,yes,no,yes,yes,no,one_year,no,electronic_check
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5629,male,1,no,no,yes,yes,fiber_optic,no,no,yes,no,yes,yes,month-to-month,yes,electronic_check
5630,male,0,no,yes,yes,no,no,no_internet_service,no_internet_service,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check
5631,male,0,no,no,yes,yes,fiber_optic,no,yes,yes,no,yes,yes,month-to-month,yes,electronic_check
5632,male,0,no,no,yes,yes,dsl,no,yes,no,no,no,no,month-to-month,yes,mailed_check


# Feature importance
- Evaluate churn rate within different groups

In [20]:
round(df_full_train.groupby('gender').churn.mean(), 2)

gender
female    0.28
male      0.26
Name: churn, dtype: float64

In [21]:
round(df_full_train.groupby('partner').churn.mean(), 2)

partner
no     0.33
yes    0.21
Name: churn, dtype: float64

In [29]:
from IPython.display import display

In [30]:
for col in categorical_variables:
    print(col)
    df_group =  df_full_train.groupby(col).churn.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_churn_rate
    df_group['risk_rate'] = df_group['mean'] / global_churn_rate
    display(df_group)
    print()
    

gender


,mean,count,diff,risk_rate
gender,,,,
female,0.276824,2796,0.006824,1.025274
male,0.263214,2838,-0.006786,0.974865



seniorcitizen


,mean,count,diff,risk_rate
seniorcitizen,,,,
0,0.242270,4722,-0.027730,0.897297
1,0.413377,912,0.143377,1.531027



partner


...

,mean,count,diff,risk_rate
contract,,,,
month-to-month,0.431701,3104,0.161701,1.598893
one_year,0.120573,1186,-0.149427,0.446568
two_year,0.028274,1344,-0.241726,0.104718



paperlessbilling


,mean,count,diff,risk_rate
paperlessbilling,,,,
no,0.172071,2313,-0.097929,0.637300
yes,0.338151,3321,0.068151,1.252412



paymentmethod


,mean,count,diff,risk_rate
paymentmethod,,,,
bank_transfer_(automatic),0.168171,1219,-0.101829,0.622854
credit_card_(automatic),0.164339,1217,-0.105661,0.608661
electronic_check,0.455890,1893,0.185890,1.688482
mailed_check,0.193870,1305,-0.076130,0.718036


### Feature importance: mutual information


In [25]:
from sklearn.metrics import mutual_info_score

In [28]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.churn)

In [33]:
mi = df_full_train[categorical_variables].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

contract            0.098320
onlinesecurity      0.063085
techsupport         0.061032
internetservice     0.055868
onlinebackup        0.046923
deviceprotection    0.043453
paymentmethod       0.043210
streamingtv         0.031853
streamingmovies     0.031581
paperlessbilling    0.017589
dependents          0.012346
partner             0.009968
seniorcitizen       0.009410
multiplelines       0.000857
phoneservice        0.000229
gender              0.000117
dtype: float64

### Feature importance: correlation
- correlation coefficient


In [35]:
df_full_train[numerical_variables].corrwith(df_full_train.churn).abs()

tenure            0.351885
monthlycharges    0.196805
totalcharges      0.196353
dtype: float64

# One-hot encoding
- use scikit learn
  

In [36]:
from sklearn.feature_extraction import DictVectorizer

In [39]:
dv = DictVectorizer(sparse=False) 

In [54]:
train_dict = df_train.to_dict(orient='records') #convert the data frame into a dictionary
test_dict = df_test.to_dict(orient='records')
val_dict = df_val.to_dict(orient='records')

In [55]:
dv.fit(train_dict) #fit the data before transforming
X_train = dv.transform(train_dict)
X_test = dv.transform(test_dict)
X_val = dv.transform(val_dict)    

# Logistic regression
- Train a model with scikit-learn
- Apply it to the validation dataset
- Calculate the accuracy
  